In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv('../data/processed/features_with_rul.csv')

X = dataset.drop(columns=['engine_id', 'RUL'])
y = dataset['RUL']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("RUL range:", y.min(), "to", y.max())

X shape: (100, 777)
y shape: (100,)
RUL range: 127 to 361


In [2]:
from sklearn.feature_selection import VarianceThreshold

# Remove features with near-zero variance
selector_var = VarianceThreshold(threshold=0.01)
X_var = selector_var.fit_transform(X)

# See what survived
surviving_cols = X.columns[selector_var.get_support()]
print(f"Before: {X.shape[1]} features")
print(f"After variance filter: {X_var.shape[1]} features")
print(f"Dropped: {X.shape[1] - X_var.shape[1]} features")

X_var_df = pd.DataFrame(X_var, columns=surviving_cols)

Before: 777 features
After variance filter: 442 features
Dropped: 335 features


c:\Users\syeda\miniconda\Lib\site-packages\sklearn\feature_selection\_variance_threshold.py:114: RuntimeWarning: Degrees of freedom <= 0 for slice.
  self.variances_ = np.nanvar(X, axis=0)


In [3]:
# Drop features that are highly correlated with each other (redundant)
corr_matrix = X_var_df.corr().abs()

# Keep upper triangle only
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Drop columns where correlation > 0.95
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
X_corr_df = X_var_df.drop(columns=to_drop)

print(f"After correlation filter: {X_corr_df.shape[1]} features")
print(f"Dropped: {len(to_drop)} redundant features")

After correlation filter: 414 features
Dropped: 28 redundant features


In [6]:
# Check how many NaNs exist
nan_counts = X_corr_df.isnull().sum()
print(f"Features with NaNs: {(nan_counts > 0).sum()}")
print(f"Total NaN values: {nan_counts.sum()}")

# Fill NaNs with column median (more robust than mean)
X_corr_df = X_corr_df.fillna(X_corr_df.median())

# Verify
print(f"NaNs after fix: {X_corr_df.isnull().sum().sum()}")

Features with NaNs: 140
Total NaN values: 2588
NaNs after fix: 0


In [7]:
from sklearn.feature_selection import mutual_info_regression

# Score each feature by how much information it shares with RUL
mi_scores = mutual_info_regression(X_corr_df, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X_corr_df.columns).sort_values(ascending=False)

# Keep top 50 features
top_50 = mi_series.head(50).index
X_filtered = X_corr_df[top_50]

print(f"After mutual information filter: {X_filtered.shape[1]} features")
print("\nTop 10 most informative features:")
print(mi_series.head(10))

After mutual information filter: 50 features

Top 10 most informative features:
value__sum_values                                1.762354
value__number_peaks__n_50                        0.642392
value__fft_coefficient__attr_"abs"__coeff_1      0.594590
value__fft_coefficient__attr_"abs"__coeff_95     0.532608
value__fft_coefficient__attr_"real"__coeff_95    0.517726
value__fft_coefficient__attr_"imag"__coeff_84    0.516937
value__fft_coefficient__attr_"imag"__coeff_1     0.511290
value__fft_coefficient__attr_"abs"__coeff_10     0.507842
value__fft_coefficient__attr_"abs"__coeff_93     0.502595
value__fft_coefficient__attr_"abs"__coeff_90     0.501962
dtype: float64


In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save filtered features
X_filtered.to_csv('../data/processed/features_filtered.csv', index=False)
y.to_csv('../data/processed/rul_labels.csv', index=False)

print("Saved filtered features:", X_filtered.shape)
print("Saved RUL labels:", y.shape)

Saved filtered features: (100, 50)
Saved RUL labels: (100,)


In [9]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Load filtered features (in case kernel was restarted)
X_filtered = pd.read_csv('../data/processed/features_filtered.csv')
y = pd.read_csv('../data/processed/rul_labels.csv').squeeze()

N_FEATURES = X_filtered.shape[1]  # 50
print("Number of features:", N_FEATURES)

# Define fitness: minimize (lower RMSE + fewer features = better)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

Number of features: 50


In [10]:
def evaluate(individual):
    # Get indices of selected features (where chromosome = 1)
    selected = [i for i, bit in enumerate(individual) if bit == 1]
    
    # Must select at least 2 features
    if len(selected) < 2:
        return (9999,)
    
    # Train quick Random Forest on selected features only
    X_sel = X_filtered.iloc[:, selected]
    model = RandomForestRegressor(n_estimators=20, random_state=42)
    
    # Cross-validated RMSE
    scores = cross_val_score(model, X_sel, y, 
                             cv=5, scoring='neg_root_mean_squared_error')
    rmse = -scores.mean()
    
    # Penalty for using too many features
    penalty = 0.5 * len(selected)
    
    return (rmse + penalty,)

In [11]:
import time

# Register GA components
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, 
                 creator.Individual, toolbox.attr_bool, N_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run
random.seed(42)
pop = toolbox.population(n=30)       # 30 chromosomes
NGEN = 20                             # 20 generations

print("Starting Genetic Algorithm...")
ga_start = time.time()

result, log = algorithms.eaSimple(
    pop, toolbox,
    cxpb=0.7,      # 70% crossover probability
    mutpb=0.2,     # 20% mutation probability
    ngen=NGEN,
    verbose=True
)

ga_time = time.time() - ga_start
print(f"\nGA done in {ga_time:.1f} seconds")

# Get best chromosome
best = tools.selBest(result, k=1)[0]
selected_indices = [i for i, bit in enumerate(best) if bit == 1]
selected_features = X_filtered.columns[selected_indices].tolist()

print(f"\nSelected {len(selected_features)} features:")
print(selected_features)

Starting Genetic Algorithm...
gen	nevals
0  	30    
1  	29    
2  	28    
3  	20    
4  	24    
5  	20    
6  	18    
7  	21    
8  	17    
9  	24    
10 	20    
11 	24    
12 	23    
13 	22    
14 	22    
15 	30    
16 	18    
17 	30    
18 	23    
19 	25    
20 	23    

GA done in 94.2 seconds

Selected 8 features:
['value__sum_values', 'value__number_peaks__n_50', 'value__fft_coefficient__attr_"abs"__coeff_1', 'value__fft_coefficient__attr_"abs"__coeff_91', 'value__fft_coefficient__attr_"abs"__coeff_7', 'value__fft_coefficient__attr_"angle"__coeff_96', 'value__fft_coefficient__attr_"angle"__coeff_87', 'value__fft_coefficient__attr_"real"__coeff_96']


In [12]:
import json, os

# Save final selected feature matrix
X_final = X_filtered[selected_features]
X_final.to_csv('../data/processed/features_final.csv', index=False)

# Save selected feature names
with open('../outputs/selected_features.json', 'w') as f:
    json.dump(selected_features, f, indent=2)

# Log GA runtime
with open('../outputs/runtime_log.json', 'r') as f:
    runtime_log = json.load(f)

runtime_log['ga_seconds'] = round(ga_time, 1)
runtime_log['n_features_after_filter'] = 50
runtime_log['n_features_after_ga'] = len(selected_features)
runtime_log['total_selection_seconds'] = round(ga_time, 1)

with open('../outputs/runtime_log.json', 'w') as f:
    json.dump(runtime_log, f, indent=2)

print("Saved final features:", X_final.shape)
print("\nFull runtime log:")
print(json.dumps(runtime_log, indent=2))

Saved final features: (100, 8)

Full runtime log:
{
  "feature_extraction_seconds": 202.4,
  "n_engines": 100,
  "n_sensors": 14,
  "n_features_raw": 777,
  "n_features_per_engine": 777,
  "ga_seconds": 94.2,
  "n_features_after_filter": 50,
  "n_features_after_ga": 8,
  "total_selection_seconds": 94.2
}


## Notebook 03 — Feature Selection

### Stage 1: Filter Methods
- Variance threshold (0.01): 777 → 442 features
- Correlation filter (0.95): 442 → 414 features  
- Mutual information (top 50): 414 → 50 features

### Stage 2: Genetic Algorithm (DEAP)
- Population: 30 chromosomes, 20 generations
- Fitness: RMSE + 0.5 × feature count
- Runtime: 94.2 seconds
- Result: 8 features selected from 50